# Exercice 16 - Producer Kafka Python

## Objectifs
- Creer un producer Kafka robuste
- Envoyer des messages JSON structures
- Utiliser les cles de partitionnement
- Simuler un flux de donnees en temps reel

---

## 1. Architecture Producer

```
+------------------------------------------------------------------+
|                    PRODUCER KAFKA                                |
+------------------------------------------------------------------+
|                                                                  |
|   APPLICATION           PRODUCER              KAFKA              |
|                                                                  |
|  +-----------+      +-------------+      +----------------+      |
|  |           |      |             |      |                |      |
|  |  Donnees  |----->| Serializer  |----->|    Topic       |      |
|  |  (dict)   |      | (JSON)      |      |                |      |
|  +-----------+      +-------------+      | +------------+ |      |
|                            |             | | Partition 0| |      |
|                     +------v------+      | +------------+ |      |
|                     |             |      | | Partition 1| |      |
|                     | Partitioner |----->| +------------+ |      |
|                     | (par cle)   |      | | Partition 2| |      |
|                     +-------------+      | +------------+ |      |
|                                          |                |      |
|                                          +----------------+      |
|                                                                  |
+------------------------------------------------------------------+

Options importantes :
- bootstrap_servers : liste des brokers
- key_serializer    : serialisation des cles
- value_serializer  : serialisation des valeurs
- acks              : garantie de livraison (0, 1, all)
- retries           : nombre de tentatives en cas d'erreur
```

## 2. Configuration

In [65]:
!pip install kafka-python -q

from kafka import KafkaProducer, KafkaAdminClient
from kafka.admin import NewTopic
import json
import time
import random
from datetime import datetime

KAFKA_BROKER = "broker:29092"

print("Configuration prete")

Configuration prete


In [66]:
# Creer les topics necessaires
def creer_topic_si_absent(nom, partitions=3):
    try:
        admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
        if nom not in admin.list_topics():
            admin.create_topics([NewTopic(name=nom, num_partitions=partitions, replication_factor=1)])
            print(f"Topic '{nom}' cree")
        else:
            print(f"Topic '{nom}' existe deja")
        admin.close()
    except Exception as e:
        print(f"Erreur : {e}")

creer_topic_si_absent("commandes-json", 3)
creer_topic_si_absent("logs-application", 1)
creer_topic_si_absent("metrics", 3)

Topic 'commandes-json' cree
Topic 'logs-application' cree
Topic 'metrics' cree


## 3. Producer JSON

In [67]:
# Creer un producer avec serialisation JSON
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    acks='all',  # Attendre confirmation de tous les replicas
    retries=3,   # Reessayer 3 fois en cas d'erreur
)

print("Producer JSON configure")

Producer JSON configure


In [68]:
# Envoyer un message JSON
commande = {
    "order_id": 1001,
    "customer_id": "CUST-123",
    "timestamp": datetime.now().isoformat(),
    "produits": [
        {"product_id": 1, "quantity": 2, "price": 29.99},
        {"product_id": 5, "quantity": 1, "price": 49.99}
    ],
    "total": 109.97
}

# Utiliser customer_id comme cle (meme client = meme partition)
future = producer.send(
    topic="commandes-json",
    key=commande["customer_id"],
    value=commande
)

result = future.get(timeout=10)
print(f"Message envoye :")
print(f"  Topic: {result.topic}")
print(f"  Partition: {result.partition}")
print(f"  Offset: {result.offset}")
print(f"  Cle: {commande['customer_id']}")

Message envoye :
  Topic: commandes-json
  Partition: 1
  Offset: 0
  Cle: CUST-123


## 4. Generateur de commandes aleatoires

In [69]:
# Donnees de reference
PRODUITS = [
    {"id": 1, "nom": "Laptop", "prix": 999.99},
    {"id": 2, "nom": "Souris", "prix": 29.99},
    {"id": 3, "nom": "Clavier", "prix": 79.99},
    {"id": 4, "nom": "Ecran", "prix": 349.99},
    {"id": 5, "nom": "Casque", "prix": 149.99},
    {"id": 6, "nom": "Webcam", "prix": 89.99},
    {"id": 7, "nom": "USB Hub", "prix": 39.99},
    {"id": 8, "nom": "SSD", "prix": 129.99},
]

CLIENTS = [f"CUST-{i:03d}" for i in range(1, 21)]  # 20 clients

def generer_commande():
    """Genere une commande aleatoire"""
    client = random.choice(CLIENTS)
    nb_produits = random.randint(1, 4)
    
    produits_commande = []
    total = 0.0
    
    for _ in range(nb_produits):
        produit = random.choice(PRODUITS)
        quantite = random.randint(1, 3)
        sous_total = produit["prix"] * quantite
        
        produits_commande.append({
            "product_id": produit["id"],
            "product_name": produit["nom"],
            "quantity": quantite,
            "unit_price": produit["prix"],
            "subtotal": round(sous_total, 2)
        })
        total += sous_total
    
    return {
        "order_id": random.randint(10000, 99999),
        "customer_id": client,
        "timestamp": datetime.now().isoformat(),
        "items": produits_commande,
        "total": round(total, 2),
        "status": "created"
    }

In [70]:
# Tester le generateur
commande_test = generer_commande()
print(json.dumps(commande_test, indent=2))

{
  "order_id": 22189,
  "customer_id": "CUST-020",
  "timestamp": "2026-03-27T11:08:01.856999",
  "items": [
    {
      "product_id": 1,
      "product_name": "Laptop",
      "quantity": 2,
      "unit_price": 999.99,
      "subtotal": 1999.98
    },
    {
      "product_id": 1,
      "product_name": "Laptop",
      "quantity": 1,
      "unit_price": 999.99,
      "subtotal": 999.99
    },
    {
      "product_id": 7,
      "product_name": "USB Hub",
      "quantity": 3,
      "unit_price": 39.99,
      "subtotal": 119.97
    },
    {
      "product_id": 4,
      "product_name": "Ecran",
      "quantity": 2,
      "unit_price": 349.99,
      "subtotal": 699.98
    }
  ],
  "total": 3819.92,
  "status": "created"
}


## 5. Envoi en batch

In [71]:
# Envoyer un lot de commandes
def envoyer_batch(producer, topic, nb_messages):
    """Envoie un lot de messages"""
    futures = []
    
    for i in range(nb_messages):
        commande = generer_commande()
        future = producer.send(
            topic=topic,
            key=commande["customer_id"],
            value=commande
        )
        futures.append((commande["order_id"], future))
    
    # Attendre toutes les confirmations
    producer.flush()
    
    # Verifier les resultats
    succes = 0
    erreurs = 0
    
    for order_id, future in futures:
        try:
            result = future.get(timeout=10)
            succes += 1
        except Exception as e:
            print(f"Erreur pour commande {order_id}: {e}")
            erreurs += 1
    
    return succes, erreurs

In [72]:
# Envoyer 50 commandes
print("Envoi de 50 commandes...")
succes, erreurs = envoyer_batch(producer, "commandes-json", 50)

print(f"\nResultat :")
print(f"  Succes : {succes}")
print(f"  Erreurs: {erreurs}")

Envoi de 50 commandes...

Resultat :
  Succes : 50
  Erreurs: 0


## 6. Simulation de flux en temps reel

In [73]:
def simuler_flux(producer, topic, nb_messages, delai_ms=500):
    """
    Simule un flux de donnees en temps reel.
    
    Args:
        producer: KafkaProducer
        topic: Nom du topic
        nb_messages: Nombre de messages a envoyer
        delai_ms: Delai entre les messages (millisecondes)
    """
    print(f"Simulation de {nb_messages} messages...")
    print(f"Delai entre messages: {delai_ms}ms")
    print("=" * 50)
    
    for i in range(nb_messages):
        commande = generer_commande()
        
        future = producer.send(
            topic=topic,
            key=commande["customer_id"],
            value=commande
        )
        
        result = future.get(timeout=10)
        
        print(f"[{i+1}/{nb_messages}] Order {commande['order_id']} | "
              f"Client {commande['customer_id']} | "
              f"Total {commande['total']:,.2f} EUR | "
              f"Partition {result.partition}")
        
        time.sleep(delai_ms / 1000.0)
    
    print("=" * 50)
    print("Simulation terminee")

In [74]:
# Lancer une simulation (10 messages, 300ms entre chaque)
simuler_flux(producer, "commandes-json", 10, delai_ms=300)

Simulation de 10 messages...
Delai entre messages: 300ms
[1/10] Order 66422 | Client CUST-018 | Total 1,199.95 EUR | Partition 2
[2/10] Order 28444 | Client CUST-011 | Total 359.93 EUR | Partition 2
[3/10] Order 77465 | Client CUST-007 | Total 509.95 EUR | Partition 1
[4/10] Order 10445 | Client CUST-003 | Total 849.94 EUR | Partition 1
[5/10] Order 16799 | Client CUST-012 | Total 2,259.96 EUR | Partition 2
[6/10] Order 47332 | Client CUST-011 | Total 1,369.93 EUR | Partition 2
[7/10] Order 36054 | Client CUST-007 | Total 469.96 EUR | Partition 1
[8/10] Order 40394 | Client CUST-007 | Total 3,709.92 EUR | Partition 1
[9/10] Order 31961 | Client CUST-007 | Total 2,039.97 EUR | Partition 1
[10/10] Order 80715 | Client CUST-008 | Total 7,049.91 EUR | Partition 2
Simulation terminee


## 7. Producer avec callbacks

In [75]:
# Callbacks pour gerer succes et erreurs
def on_success(record_metadata):
    """Callback appele en cas de succes"""
    print(f"[OK] Topic={record_metadata.topic}, "
          f"Partition={record_metadata.partition}, "
          f"Offset={record_metadata.offset}")

def on_error(exception):
    """Callback appele en cas d'erreur"""
    print(f"[ERREUR] {exception}")

In [76]:
# Utiliser les callbacks
for i in range(5):
    commande = generer_commande()
    
    producer.send(
        topic="commandes-json",
        key=commande["customer_id"],
        value=commande
    ).add_callback(on_success).add_errback(on_error)

producer.flush()

[OK] Topic=commandes-json, Partition=2, Offset=21
[OK] Topic=commandes-json, Partition=1, Offset=23
[OK] Topic=commandes-json, Partition=1, Offset=24
[OK] Topic=commandes-json, Partition=1, Offset=25
[OK] Topic=commandes-json, Partition=0, Offset=17


## 8. Producer de logs

In [77]:
# Generateur de logs applicatifs
NIVEAUX_LOG = ["INFO", "INFO", "INFO", "WARNING", "ERROR"]  # Plus de INFO
MODULES = ["auth", "api", "database", "cache", "payment"]
MESSAGES = {
    "auth": ["User login", "User logout", "Authentication failed", "Session expired"],
    "api": ["Request received", "Response sent", "Rate limit exceeded", "Invalid request"],
    "database": ["Query executed", "Connection opened", "Connection closed", "Query timeout"],
    "cache": ["Cache hit", "Cache miss", "Cache cleared", "Cache updated"],
    "payment": ["Payment initiated", "Payment completed", "Payment failed", "Refund processed"]
}

def generer_log():
    """Genere un log applicatif"""
    module = random.choice(MODULES)
    niveau = random.choice(NIVEAUX_LOG)
    message = random.choice(MESSAGES[module])
    
    return {
        "timestamp": datetime.now().isoformat(),
        "level": niveau,
        "module": module,
        "message": message,
        "request_id": f"req-{random.randint(10000, 99999)}",
        "user_id": f"user-{random.randint(1, 100)}",
        "duration_ms": random.randint(1, 500)
    }

In [78]:
# Envoyer des logs
print("Envoi de logs...")
print("=" * 60)

for i in range(15):
    log = generer_log()
    
    # Utiliser le niveau comme cle
    future = producer.send(
        topic="logs-application",
        key=log["level"],
        value=log
    )
    
    result = future.get(timeout=10)
    print(f"[{log['level']:7}] {log['module']:10} | {log['message']}")

producer.flush()
print("=" * 60)

Envoi de logs...
[INFO   ] api        | Rate limit exceeded
[INFO   ] cache      | Cache updated
[ERROR  ] cache      | Cache cleared
[INFO   ] cache      | Cache updated
[WARNING] cache      | Cache updated
[WARNING] auth       | User logout
[INFO   ] payment    | Payment initiated
[ERROR  ] payment    | Refund processed
[ERROR  ] cache      | Cache hit
[INFO   ] database   | Query executed
[ERROR  ] api        | Response sent
[INFO   ] cache      | Cache cleared
[INFO   ] api        | Invalid request
[WARNING] database   | Connection closed
[INFO   ] database   | Query timeout


## 9. Producer de metriques

In [79]:
def generer_metriques():
    """Genere des metriques systeme"""
    return {
        "timestamp": datetime.now().isoformat(),
        "host": f"server-{random.randint(1, 5):02d}",
        "metrics": {
            "cpu_percent": round(random.uniform(10, 95), 2),
            "memory_percent": round(random.uniform(30, 90), 2),
            "disk_percent": round(random.uniform(20, 85), 2),
            "network_in_mbps": round(random.uniform(10, 500), 2),
            "network_out_mbps": round(random.uniform(5, 200), 2),
            "requests_per_sec": random.randint(100, 5000),
            "response_time_ms": round(random.uniform(5, 100), 2)
        }
    }

# Envoyer des metriques
print("Envoi de metriques...")
for i in range(10):
    metric = generer_metriques()
    
    producer.send(
        topic="metrics",
        key=metric["host"],
        value=metric
    )
    
    print(f"{metric['host']} | CPU: {metric['metrics']['cpu_percent']}% | "
          f"MEM: {metric['metrics']['memory_percent']}%")

producer.flush()

Envoi de metriques...
server-03 | CPU: 24.55% | MEM: 48.42%
server-05 | CPU: 42.06% | MEM: 35.88%
server-05 | CPU: 14.6% | MEM: 60.05%
server-05 | CPU: 65.53% | MEM: 79.13%
server-03 | CPU: 56.81% | MEM: 77.57%
server-02 | CPU: 18.47% | MEM: 30.58%
server-04 | CPU: 35.93% | MEM: 41.7%
server-05 | CPU: 89.63% | MEM: 35.28%
server-01 | CPU: 73.06% | MEM: 63.53%
server-01 | CPU: 70.31% | MEM: 79.25%


In [80]:
# Fermer le producer
producer.close()
print("Producer ferme")

Producer ferme


---

## Exercice

**Objectif** : Creer un producer personnalise

**Consigne** :
1. Creez un topic "exercice-events"
2. Creez un generateur d'evenements utilisateur (click, view, purchase)
3. Envoyez 20 evenements avec une simulation de flux

A vous de jouer :

In [81]:
# TODO: Creer le topic
topic_name = "exercice-topic"
creer_topic_si_absent(topic_name, partitions=2)
print("\nTopic 'exercice-topic' cree pour les exercices suivants")

Topic 'exercice-topic' existe deja

Topic 'exercice-topic' cree pour les exercices suivants


In [82]:
# TODO: Creer le generateur d'evenements
def generer_event():
    # generator of random events (e.g. commandes, logs, metriques)
    return {
        "timestamp": datetime.now().isoformat(),
        "event_type": random.choice(["commande", "log", "metrique"]),
        "data": {
            "example_field": random.randint(1, 100), 
            "message": "This is a sample event", 
            "organization": random.choice(["OrgA", "OrgB", "OrgC"]),
            "organisor contact": f"contact-{random.randint(1, 100)}@example.com",
            "special_care": random.choice([True, False])
        }
        
    }
print("Generateur d'evenements configuré")

Generateur d'evenements configuré


In [83]:
# TODO: Envoyer les evenements
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    acks='all',
    retries=3,
)
print("Producer JSON configure")
print("=" * 60)

for i in range(10):
    event = generer_event()
    future = producer.send(
        topic=topic_name,
        key=event["event_type"],
        value=event
    )
    result = future.get(timeout=10)
    print(f"[{i+1}/10] Event {event['event_type']:10} | Partition: {result.partition} | Offset: {result.offset}")

# Ensure all messages are sent and close
producer.flush()
producer.close()
print("=" * 60)
print("Tous les evenements envoyes et producer ferme")

Producer JSON configure
[1/10] Event commande   | Partition: 1 | Offset: 5
[2/10] Event metrique   | Partition: 0 | Offset: 5
[3/10] Event log        | Partition: 0 | Offset: 6
[4/10] Event metrique   | Partition: 0 | Offset: 7
[5/10] Event metrique   | Partition: 0 | Offset: 8
[6/10] Event commande   | Partition: 1 | Offset: 6
[7/10] Event commande   | Partition: 1 | Offset: 7
[8/10] Event commande   | Partition: 1 | Offset: 8
[9/10] Event commande   | Partition: 1 | Offset: 9
[10/10] Event commande   | Partition: 1 | Offset: 10
Tous les evenements envoyes et producer ferme


---

## Resume

Dans ce notebook, vous avez appris :
- Comment configurer un **Producer Kafka** robuste
- Comment **serialiser en JSON**
- Comment utiliser les **cles de partitionnement**
- Comment **envoyer en batch**
- Comment **simuler un flux** en temps reel
- Comment utiliser les **callbacks**

### Prochaine etape
Dans le prochain notebook, nous apprendrons a consommer les messages avec Spark.